# Task 1 — Baseline system effectiveness
≥500 obrazów enrolled userów. Balans genuine/impostor. FAR, FRR, Accuracy.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.metrics import compute_far_frr, plot_roc, compute_roc
from src.utils import list_images
import cv2

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()

In [ ]:
# Załaduj obrazy testowe ze zbioru test/
TEST_DIR = Path('../data/test')
scores, labels = [], []

for user_dir in sorted(TEST_DIR.iterdir()):
    if not user_dir.is_dir(): continue
    user_id = user_dir.name
    if user_id not in db.get_all_users(): continue  # tylko enrolled
    for img_path in list_images(user_dir):
        img = cv2.imread(str(img_path))
        if img is None: continue
        emb = get_embedding(app, img)
        if emb is None: continue
        # genuine: max similarity z enrolled
        refs = db.get_user_embeddings(user_id)
        score = max(cosine_similarity(emb, r) for r in refs)
        scores.append(score); labels.append(1)

# TODO: dodaj pary impostor (inne osoby → label=0)

scores = np.array(scores)
labels = np.array(labels)
print(f'Próbek: {len(scores)} (genuine: {labels.sum()}, impostor: {(labels==0).sum()})')

In [ ]:
THRESHOLD = 0.4
far, frr, acc = compute_far_frr(scores, labels, THRESHOLD)
print(f'Threshold: {THRESHOLD:.2f}')
print(f'FAR:  {far:.4f} ({far*100:.2f}%)')
print(f'FRR:  {frr:.4f} ({frr*100:.2f}%)')
print(f'Acc:  {acc:.4f} ({acc*100:.2f}%)')

In [ ]:
fpr, tpr, thresholds = compute_roc(scores, labels)
fig, ax = plt.subplots()
plot_roc(fpr, tpr, ax=ax, label='Task 1 Baseline')
plt.tight_layout()
plt.savefig('../results/task1/roc.png', dpi=150)
plt.show()